# 📌 Dataiku Project Setup & Utilities

This section initializes a **connection to a Dataiku DSS project** and imports all the necessary libraries for data processing, similarity matching, and PDF parsing.

---

## 🛠 Imports

- **dataikuapi** → Connects to Dataiku DSS API.  
- **typing (Dict, Any)** → Type hints for better code readability.  
- **os, sys, io, json, uuid, logging** → System utilities, file handling, JSON parsing, unique ID generation, and logging.  
- **cosine_similarity (scikit-learn)** → Measures similarity between numeric vectors (useful for ML/NLP tasks).  
- **rapidfuzz.fuzz** → High-performance fuzzy string matching.  
- **fitz (PyMuPDF) & pdfplumber** → Parse and extract structured data from PDF files.  
- **OpensearchUtil** → Custom utility to interact with OpenSearch (for vector/text search).  
- **get_dataiku_client_and_project** → Helper function (commented out) to get client and project.

---

## ⚙ Configuration

```python



In [2]:
import dataikuapi
from typing import Dict, Any
import os
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz
import io
import fitz  # PyMuPDF
import pdfplumber
from uuid import uuid4
from soa_extraction.opensearch_utils import OpensearchUtil

import sys
import os
import fitz  # PyMuPDF
# from langgraph_utils.Info_extractor import InfoExtractorAgent
# from langgraph_utils.chat_history import SnowflakeChatMessageHistory
import json
# from langgraph_utils.file_parser import FileParser
# from langgraph_utils.digitization import main_handler
import uuid
# from langgraph_utils import creds
# from langgraph_utils.variables import PROJECT_NAME, SECRET_NAME, TOKEN_KEY
from utils.connection import get_dataiku_client_and_project
import logging



DATAIKU_HOST = "http://10.45.152.66:10000"
API_SECRET_KEY = "dkuaps-b3EsRXVjU3w4y7nd4KEwibEr04CjFPZr"          
PROJECT_NAME = "ECSGENERATION"   

# client, proj = get_dataiku_client_and_project(PROJECT_NAME, SECRET_NAME, TOKEN_KEY)
client = dataikuapi.DSSClient(DATAIKU_HOST, API_SECRET_KEY)
proj = client.get_project(PROJECT_NAME)

# 🧠 `make_llm_call` Function – LLM-Generated Edit Check Specifications

This function uses a **Large Language Model (LLM)** to generate **deterministic JSON edit-check specifications** for a given `form_name` and `field_name`.

---

## 📋 Function Overview

```python
def make_llm_call(form_name, field_name):
    ...


In [3]:
def make_llm_call(form_name,field_name):
    prompt = f"""
        You are a Edit check list specifcation specialist for Case Report Form .
        Given is an Example of how the data needs to be Generated this is a few shot example only
        `[
        vaildation_id : MVAL_DM027
        form_name : Demographics,
        'form_domain_name':DM
        'form_field_value':Country
        'variable_name': 'COUNTRY',
        'validation_logic': '(DM.COUNTRY is enterable and missing)', 'reasoning': 'Field must not be missing when enterable', 
        'action': 'prompt user with ACTION DETAILS',
        'action_details': '<query the field for missing data>',             
                ]`
        
        now you job is to create the json output for the these input fields :
        form name :{form_name} 
        field name :{field_name}
        
        Instruction:
        - Do not Expalin yourself , only json output
        - Do not change the data , generate for the form and field provided 
        - dont hallucinate 
        
    """
    prompt2 = f"""You are an Edit Check Specification Specialist for Case Report Forms (CRFs). You will receive a user input that contains two variables: form_name and field_name. Your ONLY job is to produce a JSON array of edit-check specification objects for the provided form_name and field_name. Follow these rules exactly:

1. OUTPUT FORMAT:
   - Return raw JSON only (no markdown, no code fences, no explanations, no extra text).
   - The top-level JSON must be an JSON. 
   - Generate Output for
   form name :{form_name} 
        field name :{field_name}

2. SCHEMA (each object must include exactly these keys):
   - validation_id
   - form_name
   - form_domain_name
   - form_field_value
   - variable_name
   - validation_logic
   - reasoning
   - action
   - action_details
   - source

   Do not add or remove keys.

4. DETERMINISTIC DERIVATIONS:
   - variable_name: derive by converting field_name to UPPERCASE snake_case (letters, numbers, underscores only). Example: "Date of Birth" -> "DATE_OF_BIRTH".
   - form_domain_name: map common forms (Demographics->DM, Medical History->MH, Vital Signs->VS, Adverse Event->AE, Concomitant Meds->CM, Informed Consent->IC, Physical Examination->PE, Laboratory->LB). If the form_name is not in the mapping, derive the domain by concatenating the first 2–3 letters of each significant word in the form_name and uppercasing (e.g., "Post-treatment Follow-up" -> "PTF").
   - validation_id: deterministic string "MVAL_{{form_name}}NNN" where NNN is a 3-digit sequence starting at 001. Use 001 unless other context is provided.

... (other rules unchanged) ...

7. NO HALLUCINATION:
   - Do not invent facts, values, mappings, or external knowledge not produced by the deterministic rules above.
   - If you cannot deterministically choose a validation_logic or domain from the input, do NOT invent — instead return this exact error structure (as the only element in the array):

     
       {{
         "error": "insufficient_input",
         "form_name": "{form_name}",
         "field_name": "{field_name}",
         "message": "Cannot generate validation logic deterministically for this field."
       }}
     

End of system instructions.
"""
    
    prompt3 = f"""
You are an Edit Check Specification Specialist for Case Report Forms (CRFs). You will receive a user input that contains two variables: `form_name` and `field_name`. Your ONLY job is to produce a JSON array of edit-check specification objects for the provided `form_name` and `field_name`.

Follow these rules exactly:

1. OUTPUT FORMAT:
   - Return raw JSON only (no markdown, no code fences, no explanations, no extra text).
   - The top-level JSON must be a JSON array.
   - Generate output for:
     - form name: {form_name}
     - field name:{field_name}

2. SCHEMA (Each Object Must Include Exactly These Keys):
   - validation_id
   - form_name
   - form_domain_name
   - form_field_value
   - variable_name
   - validation_logic
   - reasoning
   - action
   - action_details
   - source

   Do not add or remove keys.

3. DETERMINISTIC DERIVATIONS:
   - variable_name: Convert field_name to UPPERCASE snake_case (letters, numbers, underscores only).
     Example: "Date of Birth" -> "DATE_OF_BIRTH".
   - form_domain_name:
       - Use standard mappings:
           Demographics -> DM
           Medical History -> MH
           Vital Signs -> VS
           Adverse Event -> AE
           Concomitant Meds -> CM
           Informed Consent -> IC
           Physical Examination -> PE
           Laboratory -> LB
       - If no mapping exists, derive by concatenating the first 2–3 letters of each significant word in form_name and uppercasing.
         Example: "Post-treatment Follow-up" -> "PTF".
   - validation_id: Deterministic string "MVAL_{{form_name}}NNN" where NNN is a 3-digit sequence starting at 001. Use 001 unless other context is provided.

4. VALIDATION LOGIC RULES:
   Generate one or more validation objects based on the following deterministic rules (if applicable):

   1. Missing Field Check: If the field is user-enterable, ensure it is not blank.
   2. Future Date Check: If the field is a date, ensure it is not in the future.
   3. Non-Conformance Check: If the field has controlled values (dictionary, code list), ensure the value is valid.
   4. Visibility Check:
      - If top-level Y/N field "Has the subject had any adverse events since the last visit?" = Yes, dependent fields must be visible.
      - If No, dependent fields must be hidden.
   5. Conditional Completion:
      - If top-level Y/N = Yes, all dependent fields must be filled out.
      - If No and a "Reason" field exists, ensure "Reason" is not blank.
   6. Other Specify Check: If "Other" is selected, "Other Specify" must not be blank.
   7. Chronological Validations:
      - Visit date must not be before previous visit date.
      - Visit date must not be after Disposition date.
      - AE start date must not be before Informed Consent (IC) date.
      - MH start date must not be after IC date.
      - End date must not be before its corresponding start date.
   8. Cross-Form Checks:
      - If AE or MH is related to a CM record, ensure CM start date is not before event start date.

   For each validation object, clearly specify:
   - validation_logic: The exact check being applied (deterministic wording).
   - reasoning: Why the check is necessary (data quality, protocol compliance, etc.).
   - action: What should happen when the check fails (e.g., flag, warning, hard edit).
   - action_details: Specific user-facing message.
   - source: Always set to "Edit Check Specification Rule".

5. NO HALLUCINATION:
   Do not invent facts, values, or mappings.
   If a validation cannot be generated deterministically, provide NaN
   
"""


    default_llm_model = proj.get_variables()['local'].get('default_llm_model')
        #self.default_llm_model = 'azureopenai:Azure-OpenAi:gpt-4o'
#         logging.info(f"[PlannerAgent] Initialized with LLM model: {self.default_llm_model}")
       
    llm = proj.get_llm(default_llm_model).as_langchain_llm(
        completion_settings={
        "temperature": 0,
                
                "timeout": 300,
            "max_tokens": 8192
            # 5 minutes
            }
        )
        
    output = llm.invoke(prompt3)
    return output

In [4]:
import os
import io
import fitz  # PyMuPDF
import pdfplumber
from uuid import uuid4


class historical_CRF:
    
    def __init__(self, client, proj, chunk_size=1000):
        self.proj = proj
        self.client = client
        self.s3_folder_dataset_id = proj.get_variables()['local'].get('file_upload') # change file upload 
        self.input_folder = proj.get_managed_folder(self.s3_folder_dataset_id)
        self.files = self.input_folder.list_contents()["items"]
        self.toc_page_limit = 20
        self.config = proj.get_variables()["local"]
        print(self.files)

    def historical_mapping(self, file_path,paths=[]):
        result = []

#         for file in paths:
#             path = file
#             print(path)
#             parts = path.strip('/').split('/')

#             if "Historical" in path:
                
#                 result.append({
                    
#                     "path": path,
                   
    
#                 })
        result.append(file_path)

        response = []

        for i in result:
            

            with self.input_folder.get_file(file_path) as stream:
                file_bytes = stream.raw.data
            
            
            try:
                
                import re

                def clean_summary(text):
                    # Remove newlines, tabs, and collapse extra spaces
                    text = re.sub(r'\s+', ' ', text).strip()

                    # Ensure it ends with a single period
                    if not text.endswith('.'):
                        text += '.'

                    return text
                
                pdf_file_like = io.BytesIO(file_bytes)
                with pdfplumber.open(pdf_file_like) as pdf:
                    
                    for page_num, page in enumerate(pdf.pages):
                        res = {}
#                         parts = i["path"].strip('/').split('/')
                        therapeutic_area = ''
                        source =  "Unknown"
                        template_name = os.path.basename(file_path)
                        unique_id = uuid4()
#                         print(hist_id)
                        res = {
                           
                            
                            "template_name": template_name,
                            "path": file_path,
                            "id": unique_id,

                        }
                        
                        page_text = page.extract_text(layout=True)
                        
                        if page_text and "field name" in page_text.lower():
                            continue

                        if not page_text:
                            continue

                        lines = page_text.split('\n')
                        header_lines = []
                        field_value_map = {}
                        current_field = ""
                        in_field_section = False

                        for line in lines:
                            line = line.strip()
                            if not line:
                                continue

                            # Trigger point for header vs fields
                            if not in_field_section:
                                if "generated" in line.lower():
                                    in_field_section = True
                                    continue
                                header_lines.append(line)
                            else:
                                if len(line.strip()) == 0:
                                    continue
                                
                                pattern = r'''
                                    ^                              # Start of line
                                    (?P<field>.+?)                 # Field name (non-greedy)
                                    (?:\t|\s{2,})+                 # Separator: tab or ≥2 spaces
                                    (?P<value>.+?)                 # Value
                                    \s*$                           # Optional trailing spaces
                                '''
                                pattern2  = r'^(?!\s)(?!.*\s$)(?P<value>.+)$'

                                
                                
                                
                                
                                
#                                 current_field = None

                                # Step 1 ─ collect every “proper” field line
                                m = re.match(pattern, line, re.VERBOSE)
                                p = re.match(pattern2, line, re.VERBOSE) 
                                if m:
                                    if m.group("field") and m.group("value"):
                                        feild = m.group("field")
                                        current_field = feild
                                        value = m.group("value")
                                        
                                        
                                if p:
                                    if p.group("value") and current_field:
                                        # Continuation of previous field
#                                         feild = current_field
                                        value = p.group("value")
                                        
                        
                               
                                
                                left_part = current_field.strip()
                                right_part = value.strip()

                                if left_part:
                                    if left_part in field_value_map and right_part:
                                        field_value_map[left_part].append(right_part)
                                     
                                    else:
                                        
                                        field_value_map[left_part] = [right_part.strip()]
                             

                        final_feilds = []
                        for k in field_value_map:
                            
                            final_feilds.append({
                                "field_name" : k,
                                "field_value": field_value_map[k]
                            })
                        
                        head = ""
                        for j in header_lines:
                            if "form" in j.lower().strip() or "folder" in j.lower().strip():
                                head += j + " "
                        
                        res["source_data"] = {
                            "assessments" : head,
                            "feilds" : final_feilds 
                            
                        }
#                         print(res)
                    
#                     print(res)
                        if final_feilds and head:
                            response.append(res)
#                             print(f"✅ extraction for {i['path']} completed")
                    

            except Exception as e:
                print(f"Error processing file {i['path']}: {e}")
        
        

        return response


 # 📑 CRF Digitization & Extraction of Form/Field Names

This section performs **digitization of a historical CRF (Case Report Form) PDF**, extracts form names and field names, and organizes them into a structured dataset for downstream processing.

---




In [5]:
obj = historical_CRF(client , proj)
file_path = '/Annotated_Otsuka_405 201 00157_00150405_v1.0_Complete eCRF (1).pdf'
import time
import re
start = time.time()
final_list = []
response = obj.historical_mapping(file_path)

for resp in response:
    form_name = resp['source_data']['assessments']
    match = re.search(r'Form[:\s]*(.*)', form_name)
    if match:
#         print(match.group(1))
        form_name = match.group(1)
    for field in resp['source_data']['feilds']:
        field_name = field['field_name']
        final_list.append({
            "form_name":form_name,
            "field_name":field_name
        })
end = time.time()
tt = end-start
import pandas as pd 
pd.set_option("display.max_rows",None)
data = pd.DataFrame(final_list)
sub_data = data.iloc[:101]
print(tt)

[{'path': '/3', 'size': 630, 'lastModified': 1758528240000}, {'path': '/Annotated_Otsuka_405 201 00157_00150405_v1.0_Complete eCRF (1).pdf', 'size': 2544020, 'lastModified': 1757496080000}, {'path': '/output.xlsx', 'size': 671576, 'lastModified': 1757925327000}, {'path': '/soatest1.pdf', 'size': 1178968, 'lastModified': 1758527785000}, {'path': '/standardECS.xlsx', 'size': 267281, 'lastModified': 1758524961000}, {'path': '/standard_ECS_output.xlsx', 'size': 267282, 'lastModified': 1758524997000}]
39.63871788978577


# 🔍 Hybrid Search over OpenSearch with LLM Fallback

This section performs a **hybrid search** over an OpenSearch index to find the most relevant **form-field mappings** for extracted CRF data.  
If the similarity score is below a defined threshold, it falls back to an **LLM-based generator** to produce deterministic specifications.

---

## 🛠 Workflow Overview

### **1. OpenSearch Setup**
- Initializes an `OpensearchUtil` client using the connected Dataiku project.
- Retrieves the OpenSearch index name from project variables and replaces any `${projectKey}` placeholders with the actual project key.

---

### **2. Iterating over Extracted CRF Data**
- Loops through each `(form_name, field_name)` pair from `sub_data` (first 101 rows of digitized CRF data).
- Generates **vector embeddings** for both `form_name` and `field_name` using the default embedding model defined in project variables.

---

### **3. Hybrid Query Construction**
- Builds an OpenSearch **kNN (vector) search query**:
  - Matches on `form_name_vector` and `form_field_value_vector`.
  - Retrieves the top 10 nearest neighbors for each vector.

---

### **4. Search Execution & Result Scoring**
- Executes the query and collects candidate hits.
- For each hit:
  - Computes **cosine similarity** between field vectors.
  - Computes **fuzzy text similarity** using `rapidfuzz`.
  - Combines them into a **hybrid score** using weighted average:  
    \[
    \text{final_score} = 0.7 \times \text{cosine_similarity} + 0.3 \times \text{text_similarity}
    \]
- Selects the hit with the highest combined score as the **best match**.

---

### **5. Threshold-Based LLM Fallback**
- If the best match's **normalized OpenSearch score** `< 0.64` →  
  Calls `make_llm_call(form_name, field_name)` to generate an **LLM-based edit-check specification**.
- Even if the top hit passes the initial threshold, it **re-checks field-level similarity**:
  - If the recomputed field-level similarity `< 0.64`, it again calls the LLM.
  - Otherwise, it keeps the OpenSearch result as-is.

---

### **6. Enriching Results**
- For LLM-generated results:
  - Adds metadata (ecs_id, form_id, original form & field names, similarity scores).
  - Marks the source as **"LLM Generated"** for traceability.
- For OpenSearch results:
  - Updates the `form_field_value` to the best-matched value and stores it.

---

### **7. Final Output**
- All results (both OpenSearch and LLM-generated) are collected into `output_list`.
- Converted to a Pandas DataFrame (`final_df2`) for further processing or review.
- Execution time is printed for performance monitoring.

---

## ✅ **Key Points**
- **Hybrid similarity** = Cosine similarity (semantic) + Fuzzy match (string-based).
- **LLM Fallback**:
  - Triggered if similarity < **0.64** (either overall hit score or field-level score).
  - Ensures robust, deterministic edit-check specification generation even when OpenSearch recall is weak.
- **Traceability**: All outputs include metadata (`source`, `score`, `field_score`) to distinguish model-generated vs. search-retrieved results.

---




In [8]:
# Single search over OpenSearch index with hybrid similarity (vector + fuzzy)
import time
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz  # for fuzzy text similarity

start = time.time()

# Initialize OpenSearch client
opensearch_client = OpensearchUtil(client, proj)
client_os = opensearch_client.opensearch_client

output_list = []

# Get the OpenSearch index name from project variables
index_name = proj.get_variables()['local'].get('ecs_opensearch')
dataiku_project_var = "${projectKey}"
if dataiku_project_var in index_name:
    # Replace projectKey placeholder with actual project key
    index_name = index_name.replace(dataiku_project_var, opensearch_client.project.project_key).lower()

# Iterate over first 10 rows of the input dataframe
for index, row in sub_data.iterrows():
    form_name = row['form_name']
    field_name = row['field_name']

    # Generate embeddings for form name and field name
    form_emb = opensearch_client.create_embedding(form_name, proj.get_variables()['local'].get("default_embeddings_model_id"))
    field_emb = opensearch_client.create_embedding(field_name, proj.get_variables()['local'].get("default_embeddings_model_id"))

    form_vec = form_emb['response']
    field_vec = field_emb['response']

    # Build OpenSearch query using vector search on form_name and field_value
    query = {
        "query": {
            "bool": {
                "should": [
                    {
                        "knn": {
                            "form_name_vector": {
                                "vector": form_vec,
                                "k": 10  # top-k candidates from ANN search
                            }
                        }
                    },
                    {
                        "knn": {
                            "form_field_value_vector": {
                                "vector": field_vec,
                                "k": 10
                            }
                        }
                    }
                ]
            }
        }
    }

    # Execute search
    p = opensearch_client.opensearch_client
    result = p.search(index=index_name, body=query, size=5)

    # Select best matching hit using hybrid score (cosine similarity + fuzzy text match)
    max_similarity = -1  # track highest combined score
    field_name_val_ = ''
    final_value = None

    for hit in result["hits"]["hits"]:
        # Compute cosine similarity between field name vector and hit vector
        cos_sim = cosine_similarity([field_vec], [hit['_source']['form_field_value_vector']])[0][0]
        # Compute fuzzy text similarity (0-1)
        text_score = fuzz.token_sort_ratio(field_name, hit['_source']['form_field_value']) / 100
        # Weighted combination (tune 0.7 / 0.3 if needed)
        final_score = 0.7 * cos_sim + 0.3 * text_score

        # Keep track of best match
        if final_score > max_similarity:
            max_similarity = final_score
            if hit["_source"]['form_name'].lower().strip() == "informed consent":
                # Debug log for informed consent hits
                print("value", hit["_source"]['form_name'], hit['_source']['form_field_value'])
                print("score", final_score,
                      f"original {form_name} ,  orginal field {field_name}",
                      hit["_source"]['form_name'], hit['_source']['form_field_value'])

            field_name_val_ = hit['_source']['form_field_value']
            final_value = hit

    # If we found a candidate hit, enrich it with original context
    if final_value:
        final_value['_source']['original_form_name'] = form_name
        final_value['_source']['original_field'] = field_name
        final_value['_source']['score'] = final_value['_score'] / 2  # normalize OpenSearch score

    # If match score is low → fallback to LLM to generate a better guess
    if final_value and final_value['_score'] / 2 < 0.64:
        print('llm call')  # debug marker
        output = json.loads(make_llm_call(form_name, field_name))
        if isinstance(output, list):
            output = output[0]  # use first LLM response

        # Add extra metadata for traceability
        output['ecs_id'] = final_value['_source']['ecs_id']
        output['form_id'] = final_value['_source']['form_id']
        output['original_form_name'] = form_name
        output['original_field'] = field_name
        output['score'] = final_value['_score'] / 2
        output['source'] = 'LLM Generated'
        output_list.append(output)
    else:
        # Use the highest similarity hit
        if final_value:
            final_score_field = 0 
            standard_emb = final_value['_source']['form_field_value_vector']
            field_embdding = opensearch_client.create_embedding(field_name,proj.get_variables()['local'].get("default_embeddings_model_id"))
            field_vec_ = field_embdding['response']
            cos_sim_field = cosine_similarity([field_vec_], [final_value['_source']['form_field_value_vector']])[0][0]
            text_score_ = fuzz.token_sort_ratio(field_name, final_value['_source']['form_field_value']) / 100
            # Weighted combination (tune 0.7 / 0.3 if needed)
            final_score_field = 0.7 * cos_sim_field + 0.3 * text_score_
            
            if final_score_field < 0.64:
                output = json.loads(make_llm_call(form_name, field_name))
                if isinstance(output, list):
                    output = output[0]  # use first LLM response

                # Add extra metadata for traceability
                output['ecs_id'] = final_value['_source']['ecs_id']
                output['form_id'] = final_value['_source']['form_id']
                output['original_form_name'] = form_name
                output['original_field'] = field_name
                output['score'] = final_value['_score'] / 2
                output['field_score'] = final_score_field
                output['source'] = 'LLM Generated'
                output_list.append(output)
            else:
            
                final_value['_source']['form_field_value'] = field_name_val_
                output_list.append(final_value['_source'])

# Convert collected results into dataframe
final_df2 = pd.DataFrame(output_list)
end = time.time()
print(f"Search completed in {end - start:.2f} seconds")


{'type': 'ElasticSearch', 'params': {'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'username': 'genai-admin', 'password': 'qPk7Jf5vcyXDS!**332gXTvSfmcauvr9', 'port': 443, 'ssl': True, 'trustAnySSLCertificate': True, 'dialect': 'ES_7', 'dkuProperties': [], 'namingRule': {'indexNameDatasetNamePrefix': '${projectKey}_'}, 'authType': 'PASSWORD', 'oauth': {'refreshTokenRotation': False}, 'aws': {'service': 'OPENSEARCH_SERVERLESS', 'credentialsMode': 'KEYPAIR', 'customAWSCredentialsProviderParams': []}}, 'credentialsMode': 'GLOBAL', 'proxySettingsAsString': ''}
opensearch <OpenSearch([{'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'port': 443}])>
llm call


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/opensearchpy/connection/http_urllib3.py:214: UserWarning: Connecting to https://aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com:443 using SSL with verify_certs=False is insecure.
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/pydantic/_internal/_config.py:341: UserWarning: Valid config keys have changed in V2:
* 'underscore_attrs_are_private' has been removed
  warnings.warn(message, UserWarning)
/data/dataiku/dss_data/code-e

llm call


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


llm call


...

llm call


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


llm call


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


llm call
Search completed in 555.56 seconds


In [9]:
final_df2

,validation_id,form_name,form_domain_name,form_field_value,variable_name,validation_logic,reasoning,action,action_details,source,ecs_id,form_id,original_form_name,original_field,score,path,form_name_vector,form_field_value_vector,field_score
0,MVAL_Enrollment001,Enrollment,EN,Site ID,SITE_ID,Ensure Site ID is not blank,Site ID is a critical identifier and must be p...,Hard Edit,Site ID cannot be left blank. Please enter a v...,LLM Generated,5322d11a-79e3-49a7-838c-0469661a9d4d,3c36705a-c794-48a0-aa54-0453ea51bde5,Enrollment,Site ID,0.312996,NaN,NaN,NaN,NaN
1,MVAL_Enrollment001,Enrollment,EN,Participant ID,PARTICIPANT_ID,Ensure PARTICIPANT_ID is not blank,Participant ID is a critical identifier and mu...,Hard Edit,Participant ID cannot be left blank. Please en...,LLM Generated,8eeef31a-97fa-4fef-bcb7-a4b1c72b1369,4a5552de-4909-4846-bd25-3c28f9351b53,Enrollment,Participant ID,0.623769,NaN,NaN,NaN,NaN
2,MVAL_Enrollment001,Enrollment,EN,Participant Number (Derived),PARTICIPANT_NUMBER_DERIVED,Ensure the field is not blank,Participant Number is a critical identifier an...,Hard edit,Participant Number (Derived) cannot be blank. ...,LLM Generated,8eeef31a-97fa-4fef-bcb7-a4b1c72b1369,4a5552de-4909-4846-bd25-3c28f9351b53,Enrollment,Participant Number (Derived),0.622363,NaN,NaN,NaN,NaN
3,MVAL_SV010,Subject Visits,SV,Visit Date,VISDAT,(SV.VISDAT is an invalid date),Field must not be an invalid date such as 31Fe...,prompt user with ACTION DETAILS,<query the field for invalid date>,Standard,5322d11a-79e3-49a7-838c-0469661a9d4d,3c36705a-c794-48a0-aa54-0453ea51bde5,Date of Visit,Visit date,0.792303,/Standard/Copy of Otsuka Standard Edit Check S...,"[-0.027394814416766167, 0.018084557726979256, ...","[-0.020077953, 0.0069108373, -0.023404991, -0....",NaN
4,MVAL_DS_IC006,Informed Consent,DS_IC,Was informed consent obtained?,IFCOCCUR,(DS_IC.IFCOCCUR is enterable and missing),Field must not be missing when enterable,prompt user with ACTION DETAILS,<query the field for missing data>,Standard,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent obtained?,0.923950,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.002886054, 0.013917152, -0.0345182, 0.07630...",NaN
5,MVAL_DS_IC001,Informed Consent,DS_IC,Date of Consent,DSSTDAT_IC,"(DS_IC.IFCOCCUR == ""Yes"") then (DS_IC.DSSTDAT_...",If informed consent was signed then Date of Co...,DS_IC.DSSTDAT_IC is enterable,<n/a>,Standard,81661bed-0ab1-46b7-8f75-712d2402ca4b,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent date,0.842450,/Standard/Copy of Otsuka Standard Edit Check S...,"[0.02261987514793873, 0.021745149046182632, -0...","[0.037947476, 0.02801224, 0.038232155, 0.05055...",NaN
6,MVAL_Informed Consent001,Informed Consent,IC,Informed consent time,INFORMED_CONSENT_TIME,Ensure INFORMED_CONSENT_TIME is not blank,Informed consent time is a critical data point...,Hard Edit,Please enter the informed consent time,LLM Generated,3d934727-4bd1-4033-97a2-d953f5361bf0,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent time,0.823638,NaN,NaN,NaN,0.610170
7,MVAL_Informed Consent001,Informed Consent,IC,Derived date,DERIVED_DATE,Ensure DERIVED_DATE is not blank,Derived date is a critical field and must be p...,Hard edit,Derived date cannot be left blank. Please ente...,LLM Generated,62ea8058-50d8-4eb0-9b1b-275ae89847b3,bd0860d8-e7ce-4589-b23b-cbcd5931650b,Informed Consent,Derived date,0.651459,NaN,NaN,NaN,0.484138
8,MVAL_Informed Consent001,Informed Consent,IC,Informed consent version number,INFORMED_CONSENT_VERSION_NUMBER,Ensure the field is not blank,To ensure the informed consent version number ...,Hard edit,Please enter the informed consent version number,LLM Generated,fda81295-a0ab-4279-8947-ac699d5ae110,c3196559-2e4e-4c3e-bd98-afcd706447a2,Informed Consent,Informed consent version number,0.830627,NaN,NaN,NaN,0.608970
9,MVAL_Informed Consent001,Informed Consent,

In [15]:
# final_df.to_excel()
import pandas as pd
import io

# Sample DataFrame
df = final_df2

# Convert to Excel bytes
buffer = io.BytesIO()
with pd.ExcelWriter(buffer, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="Sheet1")

excel_bytes = buffer.getvalue()  # <-- This is your Excel file as bytes

obj.input_folder.put_file("/standard_ECS_output.xlsx",excel_bytes)

# Example: write to disk (optional)
with open("output_second.xlsx", "wb") as f:
    f.write(excel_bytes)


In [6]:
# Single search over OpenSearch index with hybrid similarity (vector + fuzzy) + threading
import time
import json
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from rapidfuzz import fuzz
from concurrent.futures import ThreadPoolExecutor, as_completed

start = time.time()

# Initialize OpenSearch client
opensearch_client = OpensearchUtil(client, proj)
client_os = opensearch_client.opensearch_client

# Get the OpenSearch index name from project variables
index_name = proj.get_variables()['local'].get('ecs_opensearch')
dataiku_project_var = "${projectKey}"
if dataiku_project_var in index_name:
    index_name = index_name.replace(dataiku_project_var, opensearch_client.project.project_key).lower()

output_list = []

# ---- Define worker function to process each row ----
def process_row(row):
    form_name = row['form_name']
    field_name = row['field_name']

    try:
        # Generate embeddings
        form_emb = opensearch_client.create_embedding(
            form_name, proj.get_variables()['local'].get("default_embeddings_model_id")
        )
        field_emb = opensearch_client.create_embedding(
            field_name, proj.get_variables()['local'].get("default_embeddings_model_id")
        )

        form_vec = form_emb['response']
        field_vec = field_emb['response']

        # Build query
        query = {
            "query": {
                "bool": {
                    "should": [
                        {
                            "knn": {
                                "form_name_vector": {"vector": form_vec, "k": 10}
                            }
                        },
                        {
                            "knn": {
                                "form_field_value_vector": {"vector": field_vec, "k": 10}
                            }
                        },
                    ]
                }
            }
        }

        # Execute search
        p = opensearch_client.opensearch_client
        result = p.search(index=index_name, body=query, size=5)

        max_similarity = -1
        field_name_val_ = ''
        final_value = None

        for hit in result["hits"]["hits"]:
            cos_sim = cosine_similarity(
                [field_vec], [hit['_source']['form_field_value_vector']]
            )[0][0]
            text_score = fuzz.token_sort_ratio(field_name, hit['_source']['form_field_value']) / 100
            final_score = 0.7 * cos_sim + 0.3 * text_score

            if final_score > max_similarity:
                max_similarity = final_score
                if hit["_source"]['form_name'].lower().strip() == "informed consent":
                    print("value", hit["_source"]['form_name'], hit['_source']['form_field_value'])
                    print("score", final_score,
                          f"original {form_name} , orginal field {field_name}",
                          hit["_source"]['form_name'], hit['_source']['form_field_value'])

                field_name_val_ = hit['_source']['form_field_value']
                final_value = hit

        # Enrich with original context
        if final_value:
            final_value['_source']['original_form_name'] = form_name
            final_value['_source']['original_field'] = field_name
            final_value['_source']['score'] = final_value['_score'] / 2

        # Decide between LLM or using final_value
        if final_value and final_value['_score'] / 2 < 0.64:
            print('llm call')
            output = json.loads(make_llm_call(form_name, field_name))
            if isinstance(output, list):
                output = output[0]
            output['ecs_id'] = final_value['_source']['ecs_id']
            output['form_id'] = final_value['_source']['form_id']
            output['original_form_name'] = form_name
            output['original_field'] = field_name
            output['score'] = final_value['_score'] / 2
            output['source'] = 'LLM Generated'
            return output
        elif final_value:
            # Recompute hybrid score for extra validation
            field_embdding = opensearch_client.create_embedding(
                field_name, proj.get_variables()['local'].get("default_embeddings_model_id")
            )
            field_vec_ = field_embdding['response']
            cos_sim_field = cosine_similarity(
                [field_vec_], [final_value['_source']['form_field_value_vector']]
            )[0][0]
            text_score_ = fuzz.token_sort_ratio(field_name, final_value['_source']['form_field_value']) / 100
            final_score_field = 0.7 * cos_sim_field + 0.3 * text_score_

            if final_score_field < 0.64:
                output = json.loads(make_llm_call(form_name, field_name))
                if isinstance(output, list):
                    output = output[0]
                output['ecs_id'] = final_value['_source']['ecs_id']
                output['form_id'] = final_value['_source']['form_id']
                output['original_form_name'] = form_name
                output['original_field'] = field_name
                output['score'] = final_value['_score'] / 2
                output['field_score'] = final_score_field
                output['source'] = 'LLM Generated'
                return output
            else:
                final_value['_source']['form_field_value'] = field_name_val_
                return final_value['_source']

    except Exception as e:
        print(f"Error processing row: {e}")
        return None

# ---- Run in parallel using ThreadPoolExecutor ----
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(process_row, row) for _, row in sub_data.iterrows()]
    for future in as_completed(futures):
        result = future.result()
        if result:
            output_list.append(result)

# Convert to dataframe
final_df2 = pd.DataFrame(output_list)

end = time.time()
print(f"Search completed in {end - start:.2f} seconds using threading")


{'type': 'ElasticSearch', 'params': {'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'username': 'genai-admin', 'password': 'qPk7Jf5vcyXDS!**332gXTvSfmcauvr9', 'port': 443, 'ssl': True, 'trustAnySSLCertificate': True, 'dialect': 'ES_7', 'dkuProperties': [], 'namingRule': {'indexNameDatasetNamePrefix': '${projectKey}_'}, 'authType': 'PASSWORD', 'oauth': {'refreshTokenRotation': False}, 'aws': {'service': 'OPENSEARCH_SERVERLESS', 'credentialsMode': 'KEYPAIR', 'customAWSCredentialsProviderParams': []}}, 'credentialsMode': 'GLOBAL', 'proxySettingsAsString': ''}
opensearch <OpenSearch([{'host': 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com', 'port': 443}])>


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/opensearchpy/connection/http_urllib3.py:214: UserWarning: Connecting to https://aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com:443 using SSL with verify_certs=False is insecure.
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verific

value Informed Consent Was informed consent obtained?
score 0.6409953088695635 original Informed Consent  , orginal field Informed consent date Informed Consent Was informed consent obtained?
llm call
value Informed Consent Was informed consent obtained?
score 0.842156129396492 original Informed Consent  , orginal field Informed consent obtained? Informed Consent Was informed consent obtained?
llm call
value Informed Consent Date of Consent
score 0.6569393490377782 original Informed Consent  , orginal field Informed consent date Informed Consent Date of Consent
llm call
value Informed Consent Was informed consent obtained?
score 0.6091051689641851 original Informed Consent  , orginal field Informed consent time Informed Consent Was informed consent obtained?
value Informed Consent Time of Consent
score 0.6101697493023728 original Informed Consent  , orginal field Informed consent time Informed Consent Time of Consent


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/pydantic/_internal/_config.py:341: UserWarning: Valid config keys have changed in V2:
* 'underscore_attrs_are_private' has been removed
  warnings.warn(message, UserWarning)
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(
/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urlli

value Informed Consent Was informed consent obtained?
score 0.6089704032766522 original Informed Consent  , orginal field Informed consent version number Informed Consent Was informed consent obtained?
value Informed Consent Protocol Version Number
score 0.8499907778208452 original Informed Consent  , orginal field Protocol version Informed Consent Protocol Version Number
value Informed Consent Type of Consent
score 0.31390162122073995 original Informed Consent  , orginal field Standardized disposition term Informed Consent Type of Consent
llm call


...

llm call
llm call


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


llm call


/data/dataiku/dss_data/code-envs/python/CRF_ICF_UseCase/lib/python3.9/site-packages/urllib3/connectionpool.py:1064: InsecureRequestWarning: Unverified HTTPS request is being made to host 'aos-45ff22f0a69f-gqg6rgflhsh55uefx7ljnchoiy.us-east-1.es.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


llm call
Search completed in 1807.50 seconds using threading


In [0]:
fin